# Cant change RoPE cos and sin, because it leads to change all attn the same way.

In the end of notebooks it showed, that change leads to incorrect partial rope and huge loss and perplexity.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from transformers import AutoModelForCausalLM
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
DEVICE = "cuda:0"
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    #quantization_config=quantization_config,
    #dtype=torch.bfloat16,
    device_map=DEVICE,
    # cache_dir="/glazkov-dev/.cache",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [4]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(128256, 4096)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): LlamaRotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): BlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): LlamaDecoderLayer(
        (self_attn): PositionEmbeddingsAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.

https://github.com/VainF/Torch-Pruning?tab=readme-ov-file#sparse-training-optional

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [6]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [7]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [8]:
model.train()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): EmbeddingBridge(
      (hook_in): HookPoint(name='embed.hook_in')
      (hook_out): HookPoint(name='embed.hook_out')
      (_original_component): Embedding(128256, 4096)
    )
    (layers): ModuleList(
      (0): BlockBridge(
        (hook_in): HookPoint(name='blocks.0.hook_in')
        (hook_out): HookPoint(name='blocks.0.hook_out')
        (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
        (_original_component): LlamaDecoderLayer(
          (self_attn): PositionEmbeddingsAttentionBridge(
            (hook_in): HookPoint(name='blocks.0.attn.hook_in')
            (hook_out): HookPoint(name='blocks.0.attn.hook_out')
            (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
            (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
            (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
            (hook_result): HookPoint(name='blocks.0.attn.hook_resu

We don't want norms like LlamaRMSNorm with x/std(x)*weight being pruned.

In [9]:
ignored_params = []
for name, param in model.named_parameters():
    if "norm" in name:
        ignored_params.append(param)

In [10]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input_ids=input_ids,
        #use_cache=False,
        #return_dict=True,
    ).logits #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    model,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
    unwrapped_parameters=list(zip(ignored_params, [0] * len(ignored_params)))
)


go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256, 128])
go to rope True
torch.Size([1, 256

In [11]:
print("if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.")

if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.


~~unwrapped_parameters - parameters that is not in registered model.parameters()~~
unwrapped_parameters - parameters that just nn.Parameter like weight = nn.Parameter(torch.ones(5)) instead of nn.Linear()

!Torch pruning doesnt understand semantics of channels/rows of Parameter


Note: parameters setted via self.linear = nn.Linear() through `__setattr__`  
it should be nn.Parameter()

Can afford to find pruning groups only on primitive modules like nn.Linear that directly participate in computational graph, not LlamaMLP or composite layers.

But pruning in_channels doesn't have fanout effect

In [12]:
from torch_pruning.dependency.constants import MAX_VALID_DIM
print(MAX_VALID_DIM)

18446744073709551616


In [13]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )

In [14]:
from torch_pruning.dependency.node import Node
q_proj_id = id(group[0][0].source) 
group[0]

(prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), [2, 6, 9])

rotary embedding operations:

In [25]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.5.1",
  "use_cache": true,
  "vocab_size": 128256
}

In [26]:
bridge.blocks[0].attn.q._original_component

Linear(in_features=4096, out_features=4096, bias=False)

In [27]:
bridge.blocks[0].attn.k._original_component

Linear(in_features=4096, out_features=1024, bias=False)

In [28]:
bridge.blocks[0].attn.v._original_component

Linear(in_features=4096, out_features=1024, bias=False)

That means, that concat operation - about k or v _ConcatOp_1536([0, 1024, 2048])

In [29]:
model.config.num_attention_heads * model.config.head_dim #out size of q

4096

32 * 128 = 4096 for llama

In [31]:
print("prune q_proj group")
print(group)

prune q_proj group

--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=3
[1] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _Reshape_1554(), len(idxs)=3
[2] prune_out_channels on _Reshape_1554() => prune_out_channels on _ElementWiseOp_1553(TransposeBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prune_out_channels on _Slice_1552(), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_1553(TransposeBac

In [33]:
def manually_indices_repeating(num_heads: int, head_dim: int, pruning_indices: torch.Tensor):
    all_indices = []
    for head_num in range(num_heads):
        all_indices.append(
            pruning_indices+head_num*head_dim)
    return torch.cat(all_indices)

In [34]:
#q - attn heads
#k,v - kv heads

In [35]:
idxs=[2, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
idxs = [2, 6]
repeated_idxs = manually_indices_repeating(
    bridge.model.config.num_attention_heads,
    bridge.model.config.head_dim,
    torch.tensor(idxs)
)

In [36]:
repeated_idxs

tensor([   2,    6,  130,  134,  258,  262,  386,  390,  514,  518,  642,  646,
         770,  774,  898,  902, 1026, 1030, 1154, 1158, 1282, 1286, 1410, 1414,
        1538, 1542, 1666, 1670, 1794, 1798, 1922, 1926, 2050, 2054, 2178, 2182,
        2306, 2310, 2434, 2438, 2562, 2566, 2690, 2694, 2818, 2822, 2946, 2950,
        3074, 3078, 3202, 3206, 3330, 3334, 3458, 3462, 3586, 3590, 3714, 3718,
        3842, 3846, 3970, 3974])

In [37]:
bridge.blocks[0].attn.q._original_component.out_features

4096

In [38]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q._original_component, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs.tolist() )

In [39]:
len(repeated_idxs)

64

In [40]:
group.__len__()

59

In [41]:
print(group) #todo rotate half duplicate indices?


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=64
[1] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _Reshape_1554(), len(idxs)=64
[2] prune_out_channels on _Reshape_1554() => prune_out_channels on _ElementWiseOp_1553(TransposeBackward0), len(idxs)=64
[3] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prune_out_channels on _Slice_1552(), len(idxs)=64
[4] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prun

In [42]:
import torch_pruning
def map_q_indices_to_kv(
    q_idxs,
    *,
    num_q_heads,
    num_kv_heads,
    head_dim,
):
    assert num_q_heads % num_kv_heads == 0

    repeat = num_q_heads // num_kv_heads

    mapper = torch_pruning.dependency.index_mapping._GQAIndexMapping(
        repeat=repeat,
        head_dim=head_dim,
        reverse=True,
    )

    hybrid_q_idxs = [
        torch_pruning._helpers._HybridIndex(
            idx=int(idx),
            root_idx=int(idx),
        )
        for idx in q_idxs
    ]

    mapped_hybrid_idxs = mapper(hybrid_q_idxs)

    # GQA mapping является many-to-one:
    # несколько Q-heads отображаются на одну KV-head.
    kv_idxs = sorted({
        int(mapped.idx)
        for mapped in mapped_hybrid_idxs
    })

    return kv_idxs

In [43]:
config = bridge.model.config

num_q_heads = config.num_attention_heads
num_kv_heads = config.num_key_value_heads
head_dim = config.head_dim

q_idxs = list(map(int, group[0][1]))

kv_idxs = map_q_indices_to_kv(
    q_idxs,
    num_q_heads=num_q_heads,
    num_kv_heads=num_kv_heads,
    head_dim=head_dim,
)

print("Q:", len(q_idxs), len(set(q_idxs)))
print("KV:", len(kv_idxs), len(set(kv_idxs)))
print("KV min/max:", min(kv_idxs), max(kv_idxs))

Q: 64 64
KV: 16 16
KV min/max: 2 902


In [44]:
import torch_pruning


def replace_linear_indices(
    group,
    module,
    new_idxs,
    *,
    prune_out,
):
    new_idxs = sorted(set(map(int, new_idxs)))
    matched_items = []

    for i, (dep, old_idxs) in enumerate(group):
        if dep.target.module is not module:
            continue

        if prune_out:
            correct_handler = DG.is_out_channel_pruning_fn(
                dep.handler
            )
        else:
            correct_handler = DG.is_in_channel_pruning_fn(
                dep.handler
            )

        if not correct_handler:
            continue

        print(
            f"[{i}] {dep.target.name}: "
            f"{len(old_idxs)} → {len(new_idxs)}"
        )

        group[i] = torch_pruning._helpers.GroupItem(
            dep=dep,
            idxs=new_idxs,
        )

        matched_items.append(i)

    assert len(matched_items) == 1, (
        f"Expected one group item for {module}, "
        f"found {matched_items}"
    )

In [45]:
q_proj = bridge.blocks[0].attn.q._original_component
k_proj = bridge.blocks[0].attn.k._original_component
v_proj = bridge.blocks[0].attn.v._original_component
o_proj = bridge.blocks[0].attn.o._original_component

replace_linear_indices(
    group,
    k_proj,
    kv_idxs,
    prune_out=True,
)

replace_linear_indices(
    group,
    v_proj,
    kv_idxs,
    prune_out=True,
)

[56] model.layers.0._original_component.self_attn._original_component.k_proj._original_component (Linear(in_features=4096, out_features=1024, bias=False)): 64 → 16
[39] model.layers.0._original_component.self_attn._original_component.v_proj._original_component (Linear(in_features=4096, out_features=1024, bias=False)): 64 → 16


In [46]:
# for dep, idx in group.items:
#     print(dep, idx)

In [47]:
# 3. Do the pruning
if DG.check_pruning_group(group): # avoid over-pruning, i.e., channels=0.
    group.prune()
# 4. Save & Load
# model.zero_grad() # clear gradients to avoid a large file size
# torch.save(model, 'model.pth') # !! no .state_dict here since the structure has been changed after pruning
# model = torch.load('model.pth') # load the pruned model. you may need torch.load('model.pth', weights_only=False) for PyTorch 2.6.0+.


In [48]:
len(repeated_idxs) / bridge.model.config.num_attention_heads

2.0

In [49]:
bridge.model.config.head_dim = bridge.model.config.head_dim - (len(repeated_idxs) // bridge.model.config.num_attention_heads) 

In [50]:
bridge.model.config.head_dim

126

In [51]:
bridge.blocks[0].attn._original_component.head_dim

128

Manual update of static params

In [52]:
bridge.blocks[0].attn._original_component.head_dim = bridge.model.config.head_dim

Manual rope resize

In [53]:
import torch


@torch.no_grad()
def resize_llama_rope( #dont want to use, deprecated
                      #and also goes to wrong partial rope for TransformerLens bridge
                      #if we change rope once, we should change all
                      #attentions in all layers the same way.
                      #or will get huge perplexity and loss increase.
    *,
    rotary,
    config,
    attention_modules,
    new_head_dim: int,
):
    """
    Rebuild Llama RoPE for a new attention head dimension.

    Parameters
    ----------
    rotary:
        Original HF LlamaRotaryEmbedding module.
    config:
        HF LlamaConfig shared by the model.
    attention_modules:
        Iterable of original HF LlamaAttention modules.
    new_head_dim:
        New Q/K/V head dimension after structural pruning.
    """
    if new_head_dim <= 0:
        raise ValueError(
            f"new_head_dim must be positive, got {new_head_dim}"
        )

    if new_head_dim % 2 != 0:
        raise ValueError(
            "RoPE head dimension must be even, "
            f"got {new_head_dim}"
        )

    try:
        reference_tensor = rotary.inv_freq
        device = reference_tensor.device
    except AttributeError as exc:
        raise TypeError(
            f"{type(rotary).__name__} does not expose inv_freq"
        ) from exc

    old_head_dim = getattr(config, "head_dim", None)
    old_inv_freq_shape = tuple(rotary.inv_freq.shape)

    # LlamaRotaryEmbedding reads head_dim from config.
    config.head_dim = new_head_dim

    # Recreate the same rotary class. This preserves rope_type-specific
    # initialization: default, llama3, linear, yarn, etc.
    new_rotary = type(rotary)(
        config=config,
        device=device,
    )

    # Replace buffers in-place so existing bridge/model references remain valid.
    rotary.register_buffer(
        "inv_freq",
        new_rotary.inv_freq.detach().clone(),
        persistent=False,
    )

    if hasattr(new_rotary, "original_inv_freq"):
        rotary.register_buffer(
            "original_inv_freq",
            new_rotary.original_inv_freq.detach().clone(),
            persistent=False,
        )

    if hasattr(new_rotary, "attention_scaling"):
        rotary.attention_scaling = new_rotary.attention_scaling

    if hasattr(new_rotary, "max_seq_len_cached"):
        rotary.max_seq_len_cached = new_rotary.max_seq_len_cached

    if hasattr(new_rotary, "original_max_seq_len"):
        rotary.original_max_seq_len = (
            new_rotary.original_max_seq_len
        )

    # Update every attention layer.
    for attention in attention_modules:
        attention.head_dim = new_head_dim
        attention.scaling = new_head_dim**-0.5

    expected_inv_freq_size = new_head_dim // 2

    if rotary.inv_freq.numel() != expected_inv_freq_size:
        raise RuntimeError(
            "Unexpected resized RoPE shape: "
            f"inv_freq={tuple(rotary.inv_freq.shape)}, "
            f"expected {expected_inv_freq_size} elements"
        )

    print(
        "RoPE resized:",
        f"head_dim {old_head_dim} → {new_head_dim},",
        f"inv_freq {old_inv_freq_shape} "
        f"→ {tuple(rotary.inv_freq.shape)}",
    )

    return rotary

print pruning functions & layers

In [58]:
from utils import evaluate_language_model
bridge.reset_hooks()
baseline_metrics = evaluate_language_model(
    bridge,
    evaluation_blocks,
    batch_size=EVAL_BATCH_SIZE,
)
baseline_metrics

go to rope True
torch.Size([1, 256, 126])
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to rope True
torch.Size([1, 256, 126])
partial rope!
go to 

{'loss': 3.18994140625, 'perplexity': 24.287004470825195}

We see lots of 'partial rope' messages - other layers get incompatible sizes of cos and x. 

In [ ]:
#metrics of original model
pruning_zero_metrics = {
    'loss': 2.26513671875,
    'perplexity': 9.632441520690918,
}

for idxs=[2, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

{'loss': 6.7177734375, 'perplexity': 826.97412109375}

for idx = [2, 6]  (when prune cos and sin in rope manually, not in-fly)  
{'loss': 3.18994140625, 'perplexity': 24.287004470825195}

huge increase because it wrong goes to partial rope implementation when rope.size < head_dim, it's totally wrong application of freqs and cos to part of embedding.

for idx = [2, 6] (masked rotary cos and sin for only concrete layer)  
{'loss': 2.308349609375, 'perplexity': 10.057811737060547}


Seems like correct little increase!